# 02 | Physics-Aware Exploratory Analysis

## Study objective

This notebook investigates which measured signals contain stable and physically meaningful degradation information across SOFC cells, and how these signals differ between regular and randomized-redox operating regimes.

The analysis examines temporal trends, cross-cell variability, feature relationships and regime-dependent behaviour while clearly separating direct physical observations from statistical associations and unsupported causal claims.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns

from sofc_health.features.eis_advanced import (
    drt_summary,
    fit_equivalent_circuit,
    lin_kk_diagnostics,
)

sns.set_theme(style="whitegrid", context="talk", palette="colorblind")
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
P = ROOT / "data" / "processed"
eis, iv, transient = [pd.read_parquet(P / f"{name}.parquet") for name in ("eis", "iv", "transient")]
features = pd.read_parquet(P / "features.parquet")

print("Project root:", ROOT)
print("EIS shape:", eis.shape)
print("IV shape:", iv.shape)
print("Transient shape:", transient.shape)
print("Features shape:", features.shape)
print("EIS cells:", sorted(eis["cell_id"].unique()))
print("Feature columns:", len(features.columns))

## Electrochemical reading

For EIS, $Z(\omega)=Z'(\omega)+iZ''(\omega)$. A rightward high-frequency intercept is consistent with increased ohmic resistance; arc growth can reflect polarization processes, but equivalent-circuit mechanisms are not identified by shape alone. On IV curves, $p=V|j|$ and the local slope $-dV/d|j|$ is an ASR proxy. For transients, both steady current and relaxation time may change with degradation.

In [ ]:
selected = eis.query("cell_id in ['N1', 'R1'] and assessment_index in [1, 20, 40]").copy()
selected["minus_z_imag_ohm"] = -selected["z_imag_ohm"]
g = sns.relplot(
    data=selected,
    x="z_real_ohm",
    y="minus_z_imag_ohm",
    hue="assessment_index",
    col="cell_id",
    kind="line",
    facet_kws={"sharex": False, "sharey": False},
    height=4.5,
)
g.set_axis_labels("$Z'$ / ohm", "$-Z''$ / ohm").set_titles("Cell {col_name}")

In [ ]:
long = features.melt(
    id_vars=["cell_id", "assessment_index"],
    value_vars=["tr_performance_current_a_cm2", "iv_max_power_w_cm2", "eis_polarization_proxy_ohm"],
    var_name="feature",
    value_name="value",
)
g = sns.relplot(
    data=long,
    x="assessment_index",
    y="value",
    hue="cell_id",
    col="feature",
    kind="line",
    col_wrap=1,
    height=4,
    aspect=2.2,
    facet_kws={"sharey": False},
)
g.set_axis_labels("Degradation assessment", "Measured feature")

## Advanced EIS validity gate

Lin-KK residuals test consistency before interpretation. A declared $R_0-(R_1\parallel CPE_1)$ circuit supplies a numerical comparison, while DRT resolves regularized timescales without fixing one circuit. A low residual is necessary but not sufficient for mechanistic identifiability.

In [ ]:
example_spectrum = eis.query("cell_id == 'N1' and assessment_index == 1")
advanced_eis = {
    **lin_kk_diagnostics(example_spectrum),
    **fit_equivalent_circuit(example_spectrum),
    **drt_summary(example_spectrum, num_per_decade=20, num_procs=1),
}
pd.Series(advanced_eis, name="N1 assessment 1").to_frame()

## Conclusion at this point

EIS spectra are mathematically valid for the tested example.
The simple equivalent circuit does not fully represent the spectrum.
DRT indicates multiple relaxation processes.
Performance differs substantially between cells.
Regular cells generally show rising polarization proxies.
Randomized cells appear to follow a different trend.
Absolute cross-cell comparisons must be treated carefully.

Observation:
Polarization proxy generally rises in several regular cells.

Possible interpretation:
Increasing electrochemical or transport losses.

Alternative explanations:
Temperature, fuel utilization, operating protocol or contact effects.

Evidence still required:
Cross-cell trend analysis, operating-condition comparison, DRT evolution and leakage-safe validation.

Modeling consequence:
Report regular and randomized redox regimes separately before considering a pooled model.